# 02 — Bellman Equations

## Learning Objectives
1. Understand Bellman optimality equations for V* and Q*
2. Implement value iteration using Bellman backups
3. Analyze convergence behavior and residual tracking
4. Compare synchronous, asynchronous, and prioritized VI variants

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict

np.random.seed(42)

try:
    import torch
    torch.manual_seed(42)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False
    device = 'cpu'

print(f'numpy {np.__version__}, torch={TORCH_AVAILABLE}, device={device}')


## Level 1: Bellman Backup Step

In [ ]:
# Manual 3-state MDP: states {S0, S1, S2}, actions {a0, a1}
# Transition: P[s, a, s'] = probability
P = np.array([
    [[0.7, 0.3, 0.0], [0.4, 0.6, 0.0]],   # from S0: a0, a1
    [[0.0, 0.5, 0.5], [0.0, 0.2, 0.8]],   # from S1
    [[0.0, 0.0, 1.0], [0.0, 0.0, 1.0]]    # S2 absorbing
])
R = np.array([[1.0, 0.5], [2.0, 1.5], [0.0, 0.0]])  # R[s, a]
gamma = 0.9

def bellman_backup(V, P_mat, R_mat, gamma):
    """Single Bellman backup: Q(s,a) = R(s,a) + gamma*sum_s' P(s'|s,a)*V(s')."""
    n_states, n_actions = R_mat.shape
    Q = np.zeros((n_states, n_actions))
    for s in range(n_states):
        for a in range(n_actions):
            # Expected future value under transition P[s,a,:]
            Q[s, a] = R_mat[s, a] + gamma * np.dot(P_mat[s, a], V)
    V_new = Q.max(axis=1)  # Bellman optimality: V* = max_a Q*(s,a)
    return V_new, Q

# Iterative backup from zero initialization
V = np.zeros(3)
print('Bellman backups on 3-state MDP:')
print(f'  Initial V: {V}')
for i in range(8):
    V, Q = bellman_backup(V, P, R, gamma)
    print(f'  After backup {i+1}: V = {V.round(4)}')

print(f'\nOptimal Q-table:\n{Q.round(4)}')
print(f'Optimal policy (argmax_a Q*(s,a)): {Q.argmax(axis=1)}')
print(f'Action 0 is better in S1 (higher R + good transitions).')

# Verify consistency: Q*(s,a) should satisfy Bellman equation
V_star = Q.max(axis=1)
for s in range(3):
    for a in range(2):
        expected = R[s, a] + gamma * np.dot(P[s, a], V_star)
        error = abs(Q[s, a] - expected)
        print(f'  Bellman check Q*[{s},{a}]: computed={Q[s,a]:.4f}, expected={expected:.4f}, error={error:.2e}')


## Level 2: Value Iteration Convergence

In [ ]:
def value_iteration(P_vi, R_vi, gamma=0.9, tol=1e-6, max_iter=1000):
    """Synchronous value iteration on an MDP.

    Args:
        P_vi: Transition tensor shape (n_states, n_actions, n_states).
        R_vi: Reward matrix shape (n_states, n_actions).  R[s,a] = expected immediate reward.
        gamma: Discount factor in [0, 1).
        tol: Convergence tolerance on max Bellman residual.
        max_iter: Safety cap on iterations.

    Returns:
        V_star: Optimal value array shape (n_states,).
        Q_star: Optimal Q-table shape (n_states, n_actions).
        residuals: List of max |V_new - V| per iteration.
        history: List of V snapshots at each iteration.
    """
    n_states = R_vi.shape[0]
    V = np.zeros(n_states)
    history = [V.copy()]
    residuals = []
    for i in range(max_iter):
        # Vectorized Q computation: Q[s,a] = R[s,a] + gamma * P[s,a,:] @ V
        Q = R_vi + gamma * (P_vi @ V)  # shape (n_states, n_actions)
        V_new = Q.max(axis=1)
        residual = np.max(np.abs(V_new - V))
        residuals.append(residual)
        history.append(V_new.copy())
        V = V_new
        if residual < tol:
            print(f'  Converged in {i+1} iterations, residual={residual:.2e}')
            break
    else:
        print(f'  Did not converge in {max_iter} iterations')
    return V, Q, residuals, history


def make_gridworld(size=4, gamma=0.9):
    """4x4 GridWorld: -0.01 step cost, +1.0 at goal (bottom-right)."""
    n = size * size
    n_actions = 4  # up, down, left, right
    P_gw = np.zeros((n, n_actions, n))
    R_gw = np.full((n, n_actions), -0.01)  # small step cost
    goal = n - 1
    R_gw[goal, :] = 0.0  # no cost at goal

    deltas = [(-size, 0), (size, 0), (0, -1), (0, 1)]  # up/down/left/right
    for s in range(n):
        r, c = divmod(s, size)
        for a, (dr, dc) in enumerate(deltas):
            nr, nc = r + dr, c + dc
            if 0 <= nr < size and 0 <= nc < size:
                ns = nr * size + nc
                if ns == goal:
                    R_gw[s, a] = 1.0  # goal reward overrides step cost
            else:
                ns = s  # wall: stay put
            P_gw[s, a, ns] = 1.0
    # Goal is absorbing
    P_gw[goal, :, :] = 0.0
    P_gw[goal, :, goal] = 1.0
    return P_gw, R_gw


P_gw, R_gw = make_gridworld(size=4)
print('Running value iteration on 4x4 GridWorld...')
V_opt, Q_opt, residuals, history = value_iteration(P_gw, R_gw, gamma=0.9)

# Plot convergence + optimal value heatmap
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.semilogy(residuals, color='steelblue', linewidth=2)
ax1.set_xlabel('Iteration'); ax1.set_ylabel('Max Bellman Residual (log)')
ax1.set_title('Value Iteration Convergence'); ax1.grid(True, alpha=0.3)

im = ax2.imshow(V_opt.reshape(4, 4), cmap='Blues', origin='upper')
for i in range(4):
    for j in range(4):
        s = i * 4 + j
        label = 'G' if s == 15 else f'{V_opt[s]:.2f}'
        ax2.text(j, i, label, ha='center', va='center', fontsize=9)
ax2.set_title('Optimal Value Function V*(s)'); plt.colorbar(im, ax=ax2)
plt.tight_layout(); plt.savefig('rl_02_vi_convergence.png', dpi=80, bbox_inches='tight')
plt.show()

print(f'V* max={V_opt.max():.4f}, V* min={V_opt.min():.4f}')
print(f'Converged in {len(residuals)} iterations; final residual={residuals[-1]:.2e}')


## Real-World Example 1: Q* Table Computation and Policy Arrows

In [ ]:
# Compute Q*(s,a) for GridWorld and visualize per-action heatmaps
action_names = ['Up', 'Down', 'Left', 'Right']
action_arrows = {0: (0, -0.38), 1: (0, 0.38), 2: (-0.38, 0), 3: (0.38, 0)}

# Q* is already computed from value iteration
pi_star = Q_opt.argmax(axis=1)  # optimal deterministic policy

# Per-action Q heatmaps
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for a, ax in enumerate(axes):
    Q_a = Q_opt[:, a].reshape(4, 4)
    im = ax.imshow(Q_a, cmap='viridis', origin='upper', vmin=Q_opt.min(), vmax=Q_opt.max())
    for i in range(4):
        for j in range(4):
            s = i * 4 + j
            ax.text(j, i, f'{Q_opt[s, a]:.2f}', ha='center', va='center', fontsize=7)
    ax.set_title(f'Q*(s, {action_names[a]})')
    plt.colorbar(im, ax=ax)
plt.suptitle('Q*(s,a) per action — GridWorld')
plt.tight_layout(); plt.savefig('rl_02_q_tables.png', dpi=80, bbox_inches='tight')
plt.show()

# Policy arrows visualization
fig2, ax2 = plt.subplots(figsize=(5, 5))
for s in range(16):
    row, col = divmod(s, 4)
    if s == 15:
        ax2.add_patch(plt.Rectangle((col-0.45, row-0.45), 0.9, 0.9, color='gold'))
        ax2.text(col, row, 'G', ha='center', va='center', fontsize=14, fontweight='bold')
    else:
        a_opt = pi_star[s]
        dx, dy = action_arrows[a_opt]
        ax2.annotate('', xy=(col + dx, row - dy), xytext=(col, row),
                     arrowprops=dict(arrowstyle='->', color='navy', lw=2.0))
        ax2.text(col + 0.3, row - 0.3, f'{V_opt[s]:.2f}',
                 fontsize=6, color='dimgray')

ax2.set_xlim(-0.6, 3.6); ax2.set_ylim(-0.6, 3.6)
ax2.set_xticks(range(4)); ax2.set_yticks(range(4))
ax2.set_aspect('equal'); ax2.invert_yaxis()
ax2.set_title('Optimal Policy Arrows (pi* from Q*)\nNumbers = V*(s)')
ax2.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_02_policy_arrows.png', dpi=80, bbox_inches='tight')
plt.show()

print('Optimal actions per state:')
for s in range(16):
    row, col = divmod(s, 4)
    label = 'GOAL' if s == 15 else action_names[pi_star[s]]
    print(f'  s={s:2d} (r={row},c={col}): {label:5s}  V*={V_opt[s]:.3f}')


## Real-World Example 2: Bellman Residual Tracking in Production

In [ ]:
# Track how V(s) estimates evolve across iterations for each state
# Useful for diagnosing slow convergence in production RL pipelines

key_iters = [0, 1, 5, 10, 20, len(history) - 1]
key_iters = sorted(set(min(k, len(history)-1) for k in key_iters))

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes_flat = axes.flatten()
for idx, it in enumerate(key_iters[:6]):
    V_it = history[it]
    im = axes_flat[idx].imshow(V_it.reshape(4, 4), cmap='Blues', origin='upper',
                               vmin=V_opt.min(), vmax=V_opt.max())
    for i in range(4):
        for j in range(4):
            s = i * 4 + j
            axes_flat[idx].text(j, i, f'{V_it[s]:.2f}', ha='center', va='center', fontsize=8)
    axes_flat[idx].set_title(f'V at iteration {it}')
    plt.colorbar(im, ax=axes_flat[idx])
plt.suptitle('V(s) Estimates at Different Iterations (value iteration on 4x4 GridWorld)')
plt.tight_layout(); plt.savefig('rl_02_vi_snapshots.png', dpi=80, bbox_inches='tight')
plt.show()

# Early stopping analysis: what threshold gives acceptable error?
tol_levels = [1e-2, 1e-3, 1e-4, 1e-6]
print('Early stopping analysis (value iteration on 4x4 GridWorld):')
print(f'  {"Tolerance":>10} {"Iterations":>12} {"Max V error":>14} {"Residual":>10}')
for tol in tol_levels:
    V_es, Q_es, res_es, _ = value_iteration(P_gw, R_gw, gamma=0.9, tol=tol, max_iter=500)
    max_err = np.max(np.abs(V_es - V_opt))
    print(f'  {tol:>10.0e} {len(res_es):>12d} {max_err:>14.2e} {res_es[-1]:>10.2e}')

# Track per-state value changes between iterations
state_changes = np.zeros((len(history) - 1, 16))
for k in range(len(history) - 1):
    state_changes[k] = np.abs(history[k+1] - history[k])

# Which states converge slowest?
slowest = np.argsort(state_changes.sum(axis=0))[-5:][::-1]
print(f'\nSlowest converging states (by total variation): {slowest.tolist()}')
print('States far from goal take most iterations to stabilize.')

# Visualize per-state convergence trajectories
key_states = [0, 3, 12, 14]  # corners and near-goal states
state_labels = {0: 'Start(0,0)', 3: 'TopRight(0,3)', 12: 'BotLeft(3,0)', 14: 'NearGoal(3,2)'}
fig_traj, ax_traj = plt.subplots(figsize=(10, 5))
for s in key_states:
    traj = [hist[s] for hist in history]
    ax_traj.plot(traj, label=f's={s} ({state_labels[s]})', linewidth=2)
ax_traj.axhline(0, color='black', linestyle=':', alpha=0.3)
ax_traj.set_xlabel('Iteration'); ax_traj.set_ylabel('V(s) estimate')
ax_traj.set_title('Value Estimates Over Iterations (key states)')
ax_traj.legend(); ax_traj.grid(True, alpha=0.3)
plt.tight_layout(); plt.savefig('rl_02_state_trajectories.png', dpi=80, bbox_inches='tight')
plt.show()

# Compare rate of change: state 0 (far) vs state 14 (close to goal)
changes_s0 = [abs(history[k+1][0] - history[k][0]) for k in range(len(history)-1)]
changes_s14 = [abs(history[k+1][14] - history[k][14]) for k in range(len(history)-1)]
print(f'Total variation in V(s=0) across all iters: {sum(changes_s0):.4f}')
print(f'Total variation in V(s=14) across all iters: {sum(changes_s14):.4f}')
print('s=0 varies more (must propagate reward signal from far away).')

# Residual threshold guide for practitioners
print('\nPractitioner residual guide:')
print('  tol=1e-2: OK for quick approximate planning (action changes unlikely)')
print('  tol=1e-4: Standard for most gridworld/tabular RL benchmarks')
print('  tol=1e-6: High accuracy needed for safety-critical decisions')


## Real-World Example 3: Discount Factor Analysis

In [ ]:
# Sweep gamma in {0.5, 0.9, 0.99}: observe effects on V*, convergence speed, horizon
gammas = [0.5, 0.9, 0.99]
gamma_results = {}

for g in gammas:
    V_g, Q_g, res_g, _ = value_iteration(P_gw, R_gw, gamma=g, tol=1e-6)
    gamma_results[g] = {'V': V_g, 'Q': Q_g, 'residuals': res_g,
                        'n_iter': len(res_g), 'horizon': 1.0 / (1.0 - g)}

# Convergence speed comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for g, color in zip(gammas, ['steelblue', 'darkorange', 'green']):
    res = gamma_results[g]['residuals']
    axes[0].semilogy(res, label=f'gamma={g}', color=color, linewidth=2)
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Max Bellman Residual')
axes[0].set_title('Convergence Speed vs Discount Factor')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

# V*(s=0) and V*(s=14) across gammas
for g, color in zip(gammas, ['steelblue', 'darkorange', 'green']):
    V_g_vals = gamma_results[g]['V']
    axes[1].bar([f'g={g}'], [V_g_vals[0]], color=color, alpha=0.8, label=f's=0')
axes[1].set_ylabel('V*(s=0)'); axes[1].set_title('Start-State Value vs Gamma')
axes[1].grid(True, axis='y', alpha=0.3)

# V*(s) profiles across gammas for all states
states = np.arange(16)
for g, color in zip(gammas, ['steelblue', 'darkorange', 'green']):
    axes[2].plot(states, gamma_results[g]['V'], 'o-', color=color,
                 label=f'gamma={g}', alpha=0.8, linewidth=1.5, markersize=4)
axes[2].set_xlabel('State'); axes[2].set_ylabel('V*(s)')
axes[2].set_title('V*(s) Profile vs Discount Factor')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout(); plt.savefig('rl_02_gamma_analysis.png', dpi=80, bbox_inches='tight')
plt.show()

# Summary table
print(f'  {"Gamma":>7} {"Iterations":>12} {"Horizon":>10} {"V*(s=0)":>10} {"V*(s=14)":>10}')
print('  ' + '-' * 55)
for g in gammas:
    r = gamma_results[g]
    print(f'  {g:>7.2f} {r["n_iter"]:>12d} {r["horizon"]:>10.1f} '
          f'{r["V"][0]:>10.4f} {r["V"][14]:>10.4f}')
print('\nHigher gamma -> larger V*, more iterations (contraction factor ~ gamma).')


## Comparison: Value Iteration Variants

In [ ]:
# Compare: synchronous VI, asynchronous VI (random order), prioritized sweeping

def async_value_iteration(P_vi, R_vi, gamma=0.9, tol=1e-6, max_iter=5000):
    """Asynchronous VI: update one random state per step."""
    n_states = R_vi.shape[0]
    V = np.zeros(n_states)
    residuals_max = []  # track per-sweep max residual
    for sweep in range(max_iter // n_states + 1):
        order = np.random.permutation(n_states)
        sweep_res = []
        for s in order:
            Q_s = R_vi[s] + gamma * P_vi[s] @ V  # shape (n_actions,)
            V_new_s = Q_s.max()
            sweep_res.append(abs(V_new_s - V[s]))
            V[s] = V_new_s
        residuals_max.append(max(sweep_res))
        if residuals_max[-1] < tol:
            break
    return V, residuals_max


def prioritized_sweeping(P_vi, R_vi, gamma=0.9, tol=1e-6, max_iter=2000):
    """Prioritized sweeping: always update state with largest Bellman error."""
    n_states = R_vi.shape[0]
    V = np.zeros(n_states)
    residuals = []
    for _ in range(max_iter):
        Q_all = R_vi + gamma * (P_vi @ V)  # (n_states, n_actions)
        bellman_errors = np.abs(Q_all.max(axis=1) - V)  # error per state
        s_priority = np.argmax(bellman_errors)
        max_err = bellman_errors[s_priority]
        residuals.append(max_err)
        V[s_priority] = Q_all[s_priority].max()
        if max_err < tol:
            break
    return V, residuals


np.random.seed(42)
print('Running VI variants on 4x4 GridWorld...')
V_sync, _, res_sync, _ = value_iteration(P_gw, R_gw, gamma=0.9, tol=1e-6)
V_async, res_async = async_value_iteration(P_gw, R_gw, gamma=0.9, tol=1e-6)
V_prio, res_prio = prioritized_sweeping(P_gw, R_gw, gamma=0.9, tol=1e-6)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].semilogy(res_sync, label=f'Sync VI ({len(res_sync)} iters)', color='steelblue', linewidth=2)
axes[0].semilogy(res_async, label=f'Async VI ({len(res_async)} sweeps)', color='darkorange', linewidth=2)
axes[0].semilogy(res_prio, label=f'Prioritized ({len(res_prio)} updates)', color='green', linewidth=2)
axes[0].set_xlabel('Iteration/Sweep'); axes[0].set_ylabel('Max Bellman Residual')
axes[0].set_title('VI Variant Convergence'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# Final V* error vs sync VI reference
labels = ['Sync VI', 'Async VI', 'Prioritized']
errors = [0.0, np.max(np.abs(V_async - V_sync)), np.max(np.abs(V_prio - V_sync))]
axes[1].bar(labels, errors, color=['steelblue', 'darkorange', 'green'], alpha=0.8)
axes[1].set_ylabel('Max |V - V_sync|'); axes[1].set_title('Final V* Accuracy vs Sync VI')
axes[1].grid(True, axis='y', alpha=0.3)
plt.tight_layout(); plt.savefig('rl_02_vi_variants.png', dpi=80, bbox_inches='tight')
plt.show()

print(f'Updates to converge -- Sync: {len(res_sync)*16}, Async: {len(res_async)*16}, Prio: {len(res_prio)}')
print('Prioritized sweeping needs fewest state updates when reward is sparse.')

# Verify all three methods reach the same optimal policy
pi_sync = Q_opt.argmax(axis=1)
Q_async_mat = R_gw + 0.9 * (P_gw @ V_async)
Q_prio_mat = R_gw + 0.9 * (P_gw @ V_prio)
pi_async = Q_async_mat.argmax(axis=1)
pi_prio = Q_prio_mat.argmax(axis=1)
match_async = np.mean(pi_sync == pi_async)
match_prio = np.mean(pi_sync == pi_prio)
print(f'Policy agreement -- Sync vs Async: {match_async:.1%}, Sync vs Prio: {match_prio:.1%}')

# Quantitative summary table
print('\nVariant comparison summary:')
print(f'  {"Variant":>15} {"Updates":>10} {"Final V*(s=0)":>15} {"Max error vs sync":>20}')
print('  ' + '-' * 65)
print(f'  {"Sync VI":>15} {len(res_sync)*16:>10} {V_sync[0]:>15.4f} {0.0:>20.2e}')
print(f'  {"Async VI":>15} {len(res_async)*16:>10} {V_async[0]:>15.4f} {np.max(np.abs(V_async-V_sync)):>20.2e}')
print(f'  {"Prioritized":>15} {len(res_prio):>10} {V_prio[0]:>15.4f} {np.max(np.abs(V_prio-V_sync)):>20.2e}')

# Contraction rate verification: slope of log(residual) vs iteration
# Theory: convergence rate = gamma = 0.9, so log-residual slope = log(0.9) ~ -0.105
if len(res_sync) > 5:
    log_res = np.log(np.array(res_sync) + 1e-15)
    iters = np.arange(len(log_res))
    slope, intercept = np.polyfit(iters[-len(iters)//2:], log_res[-len(log_res)//2:], 1)
    print(f'\nContraction rate analysis (sync VI):')
    print(f'  Empirical slope: {slope:.4f}')
    print(f'  Theoretical log(gamma): {np.log(0.9):.4f}')
    print(f'  Agreement: {"Yes" if abs(slope - np.log(0.9)) < 0.05 else "Approx"}')

# Which action is optimal from s=0 and how does Q* compare across actions?
s0_Q = Q_opt[0]
print(f'\nQ*(s=0) per action: { {a: round(float(s0_Q[a]),4) for a in range(4)} }')
print(f'Optimal action from s=0: {["Up","Down","Left","Right"][pi_star[0]]}')
print('All actions from s=0 eventually reach goal; Right gives highest Q* (direct path).')


## Key Takeaways

**Core idea:** Bellman equations define recursive consistency conditions: V*(s) = max_a [R(s,a) + gamma * sum_s' P(s'|s,a) V*(s')]. Value iteration repeatedly applies this backup until convergence, guaranteed by the contraction mapping theorem (rate gamma).

| Method | Update order | Updates to converge | Best for |
|--------|-------------|--------------------|---------|
| Sync VI | All states | O(n_states / iter) | Small MDPs, correctness |
| Async VI | Random state | Same wall-clock, flexible | Large state spaces |
| Prioritized | Largest error | Fewest total | Sparse reward MDPs |

**Failure modes:**
- Diverges when gamma >= 1 (no contraction) or with off-policy function approximation
- Slow convergence for small gamma gaps (gamma close to 1, many iterations)
- Numerical instability if rewards are not bounded

**Related:** [01-mdp](01-markov-decision-processes.ipynb), [03-dp-rl](03-dynamic-programming-rl.ipynb)

## Exercises

1. **Policy iteration via Bellman:** Implement policy iteration: alternate between policy evaluation (solve Bellman expectation for fixed pi) and policy improvement (greedy). Compare convergence to value iteration.
2. **Stochastic transitions:** Add slip_prob=0.1 (each action moves sideways with prob 0.1). Observe how Q* changes and whether the optimal policy differs.
3. **Obstacle rewards:** Set R=-10 for 3 obstacle states. Verify that V* and pi* correctly routes around them.
4. **Gamma bound:** Prove analytically that VI convergence rate is O(gamma^k). Verify empirically by plotting log(residual) and measuring slope.